# **Regression: AdaBoost Regressor**

## **Justification of Preprocessing Strategy**

### **Scale Invariance**
The AdaBoost Regressor utilizes shallow Decision Trees (Weak Learners) as its base estimators. Because these trees partition the continuous feature space using simple mathematical thresholds rather than geometric distance calculations, the boosting ensemble is entirely invariant to feature scaling. Standardization or Normalization will not alter the model's splitting logic. Consequently, we will train the model using the **Original, Unscaled Data** to preserve clinical interpretability and computational speed.

### **The Boosting Philosophy: Why We Do Not Transfer the Decision Tree Champion**
Unlike our Random Forest approach (where we injected a robust `max_depth=20` champion tree), we strictly avoid doing so for AdaBoost. AdaBoost relies on a sequential boosting mechanism where each iteration learns from the residual errors of the previous one. It mathematically requires "Weak Learners"—typically shallow trees with a default `max_depth` of 3. If we used a highly complex tree as the base estimator, the very first iteration would severely overfit, leaving subsequent trees with no meaningful errors to correct. Thus, we optimize only the ensemble mechanics natively.

## **Experiment Design**

We designed a tournament of 3 optimization levels. Because boosting algorithms can easily overfit if the ensemble grows too large or learns too aggressively, we explicitly log **both Train and Test metrics (RMSE, MAE, R²)** to monitor the learning gap:

* **Baseline**: Scikit-Learn defaults (`n_estimators=50`, `learning_rate=1.0`, `loss='linear'`), utilizing the default weak learner.
* **GridSearchCV**: A targeted 3-fold cross-validated search exploring the trade-off between the number of boosting stages (`n_estimators`), the contribution of each stage (`learning_rate`), and the loss function utilized to update weights.
* **Optuna Optimization**: Bayesian optimization deployed to fine-tune the continuous logarithmic space of the `learning_rate` alongside ensemble size, aiming to surgically minimize the validation RMSE without causing a spike in training memorization.

In [1]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Regression_AdaBoost")

# 2. Data Loading and Preparation
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")

categorical_cols = [
    'gender', 'ethnicity', 'smoking_status', 'education_level',
    'employment_status', 'age_groups', 'weight_status', 'income_level'
]

# Apply One-Hot Encoding
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Separate features and target 
# Drop classification targets to prevent data leakage!
X = df_final.drop(["diagnosed_diabetes", "diabetes_stage", "diabetes_risk_score"], axis=1)
y = df_final['diabetes_risk_score']

# Split data (80/20) - No stratify needed for continuous targets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

SEED = 42

def log_regression_metrics(y_tr_true, y_tr_pred, y_te_true, y_te_pred, duration):
    # Logs Train and Test metrics explicitly to monitor the Overfitting Gap
    # Train Partition Metrics
    mlflow.log_metric("rmse_train", mean_squared_error(y_tr_true, y_tr_pred) ** 0.5)
    mlflow.log_metric("mae_train", mean_absolute_error(y_tr_true, y_tr_pred))
    mlflow.log_metric("r2_train", r2_score(y_tr_true, y_tr_pred))
    
    # Test Partition Metrics
    mlflow.log_metric("rmse_test", mean_squared_error(y_te_true, y_te_pred) ** 0.5)
    mlflow.log_metric("mae_test", mean_absolute_error(y_te_true, y_te_pred))
    mlflow.log_metric("r2_test", r2_score(y_te_true, y_te_pred))
    
    mlflow.log_metric("fit_time", duration)

# ---------------------------------------------------------
# RUN 1: BASELINE
# ---------------------------------------------------------
with mlflow.start_run(run_name="AdaBoost_Reg_Baseline"):
    reg_base = AdaBoostRegressor(random_state=SEED)
    
    start_time = time.time()
    reg_base.fit(X_train, y_train)
    duration = time.time() - start_time
    
    # Explicit Predictions
    y_pred_train_base = reg_base.predict(X_train)
    y_pred_test_base = reg_base.predict(X_test)
    
    mlflow.log_params(reg_base.get_params())
    mlflow.log_param("optimization", "none_default")
    
    log_regression_metrics(y_train, y_pred_train_base, y_test, y_pred_test_base, duration)

# ---------------------------------------------------------
# RUN 2: GRIDSEARCHCV
# ---------------------------------------------------------
with mlflow.start_run(run_name="AdaBoost_Reg_GridSearch"):
    param_grid = {
        "n_estimators": [50, 100, 200],
        "learning_rate": [0.01, 0.1, 1.0],
        "loss": ["linear", "square"] # Removed 'exponential' to save time, 'linear'/'square' usually suffice
    }

    grid_reg = GridSearchCV(
        estimator=AdaBoostRegressor(random_state=SEED),
        param_grid=param_grid,
        cv=KFold(n_splits=3, shuffle=True, random_state=SEED), # 3-fold for speed on 80k rows
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    start_time = time.time()
    grid_reg.fit(X_train, y_train)
    duration = time.time() - start_time

    best_ada_grid = grid_reg.best_estimator_
    
    # Explicit Predictions
    y_pred_train_grid = best_ada_grid.predict(X_train)
    y_pred_test_grid = best_ada_grid.predict(X_test)

    mlflow.log_params(grid_reg.best_params_)
    mlflow.log_param("optimization", "GridSearchCV")
    
    log_regression_metrics(y_train, y_pred_train_grid, y_test, y_pred_test_grid, duration)

# ---------------------------------------------------------
# RUN 3: OPTUNA 
# ---------------------------------------------------------
def objective_reg(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 250),
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 1.5, log=True),
        "loss": trial.suggest_categorical("loss", ["linear", "square", "exponential"])
    }

    model = AdaBoostRegressor(**params, random_state=SEED)
    
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=KFold(n_splits=3, shuffle=True, random_state=SEED), 
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )
    return -scores.mean()

with mlflow.start_run(run_name="AdaBoost_Reg_Optuna"):
    study_reg = optuna.create_study(direction="minimize")
    
    start_time = time.time()
    study_reg.optimize(objective_reg, n_trials=12) # Safe trial budget
    duration = time.time() - start_time

    best_ada_optuna = AdaBoostRegressor(**study_reg.best_params, random_state=SEED)
    best_ada_optuna.fit(X_train, y_train)
    
    # Explicit Predictions
    y_pred_train_optuna = best_ada_optuna.predict(X_train)
    y_pred_test_optuna = best_ada_optuna.predict(X_test)

    mlflow.log_params(study_reg.best_params)
    mlflow.log_param("optimization", "optuna")
    
    log_regression_metrics(y_train, y_pred_train_optuna, y_test, y_pred_test_optuna, duration)

2026/05/22 15:18:03 INFO mlflow.tracking.fluent: Experiment with name 'Regression_AdaBoost' does not exist. Creating a new experiment.
[I 2026-05-22 15:27:39,294] A new study created in memory with name: no-name-9d85d6ee-2906-44f1-acf8-db7c9ae333e2
[I 2026-05-22 15:28:36,652] Trial 0 finished with value: 2.3070968571316834 and parameters: {'n_estimators': 196, 'learning_rate': 0.5193960176682222, 'loss': 'linear'}. Best is trial 0 with value: 2.3070968571316834.
[I 2026-05-22 15:30:05,537] Trial 1 finished with value: 3.9129163939716736 and parameters: {'n_estimators': 250, 'learning_rate': 0.0062426765568646195, 'loss': 'linear'}. Best is trial 0 with value: 2.3070968571316834.
[I 2026-05-22 15:30:50,602] Trial 2 finished with value: 3.389967486977177 and parameters: {'n_estimators': 128, 'learning_rate': 0.03871301751974873, 'loss': 'square'}. Best is trial 0 with value: 2.3070968571316834.
[I 2026-05-22 15:31:37,811] Trial 3 finished with value: 3.70286441126517 and parameters: {'n_

## Winner Run Selection (Priority Elimination Framework)

### Policy
A run is only eligible to win if it does NOT show evidence of overfitting or underfitting. Before applying the MAE/RMSE/R² decision rules, we require the Train→Test gaps to remain small enough to indicate acceptable generalization. Runs that memorize the training set or show a large Train/Test gap are disqualified regardless of metric rank.

### Selection Criteria (priority order)
1. **Generalization filter (mandatory):** runs with overfitting or underfitting are removed from consideration.
2. **Priority 1 (60%): Lowest MAE (Test)** — primary objective for regression accuracy.
3. **Priority 2 (30%): Lowest RMSE (Test)** — used to reject runs where RMSE grows disproportionately relative to MAE.
4. **Priority 3 (10%): Acceptable R² (Test)** — confirms explanatory quality.
5. **Tiebreaker: Lowest Fit Time** — if MAE, RMSE, and R² are effectively tied.

### Runs Summary

| Run | MAE (Train) | MAE (Test) | RMSE (Train) | RMSE (Test) | R² (Train) | R² (Test) | Fit Time |
|---|---:|---:|---:|---:|---:|---:|---:|
| AdaBoost_Reg_Baseline | 1.91105 | 1.92736 | 2.39009 | 2.40759 | 0.93034 | 0.92976 | 18.21s |
| AdaBoost_Reg_GridSearch | 1.39788 | 1.39646 | 1.76553 | 1.77009 | 0.96199 | 0.96203 | 552.52s |
| AdaBoost_Reg_Optuna | 1.30708 | 1.31466 | 1.67238 | 1.68522 | 0.96589 | 0.96559 | 528.38s |

### Generalization Check (Test − Train)
- **AdaBoost_Reg_Baseline:** MAE gap = 1.92736 − 1.91105 = **+0.01631** and RMSE gap = 2.40759 − 2.39009 = **+0.01751** → PASS.
- **AdaBoost_Reg_GridSearch:** MAE gap = 1.39646 − 1.39788 = **−0.00141** and RMSE gap = 1.77009 − 1.76553 = **+0.00456** → PASS.
- **AdaBoost_Reg_Optuna:** MAE gap = 1.31466 − 1.30708 = **+0.00758** and RMSE gap = 1.68522 − 1.67238 = **+0.01284** → PASS.

### Overfitting / Underfitting Validation
- None of the runs shows overfitting. The Train/Test gaps are small and stable across MAE and RMSE.
- None of the runs shows underfitting. All Test R² values are high, which indicates the ensemble is capturing the target structure well.
- There is no evidence of catastrophic RMSE growth relative to MAE.

### Step-by-Step Elimination
**Step 1 — Apply the generalization filter**
- Passing runs: all three runs.

**Step 2 — Compare Test MAE (Priority 1 — 60%)**
- AdaBoost_Reg_Optuna: 1.31466
- AdaBoost_Reg_GridSearch: 1.39646
- AdaBoost_Reg_Baseline: 1.92736
- Lowest MAE: **AdaBoost_Reg_Optuna**.

**Step 3 — Compare Test RMSE (Priority 2 — 30%)**
- AdaBoost_Reg_Optuna: 1.68522
- AdaBoost_Reg_GridSearch: 1.77009
- AdaBoost_Reg_Baseline: 2.40759
- AdaBoost_Reg_Optuna remains the best choice.

**Step 4 — Check Test R² (Priority 3 — 10%)**
- AdaBoost_Reg_Optuna: 0.96559
- AdaBoost_Reg_GridSearch: 0.96203
- AdaBoost_Reg_Baseline: 0.92976
- AdaBoost_Reg_Optuna also leads on R².

### Final Decision
**Winner: AdaBoost_Reg_Optuna**

**Justification:** `AdaBoost_Reg_Optuna` is the strongest run among those that pass the generalization filter. It has the lowest Test MAE, the lowest Test RMSE, and the highest Test R². Fit time is not needed as a tiebreaker.

## Winner Hyperparameters
| Parameter | Value |
|---|---|
| **n_estimators** | 177 |
| **learning_rate** | 1.3282165134621702 |
| **loss** | square |
| **random_state** | 42 |